# LangChain Chat Model External API Tools

**Learning pattern for every API:** define the external API tool → bind it to the chat model → ask a natural-language question → inspect the tool call → get the final answer.

All  APIs below are keyless. The Groq chat model requires a Groq API key.

APIs: Weather, currency, country information, public holidays, jokes, random users, flights and dog images.

## 1. Install packages

In [ ]:
%pip install -U langchain langchain-groq requests

## 2. Initialize the LangChain chat model FIRST

In [ ]:
import os
import requests
from getpass import getpass
from langchain.tools import tool
from langchain_groq import ChatGroq


model = ChatGroq(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b" ),  ##"qwen/qwen3.8-27b"
    api_key=os.getenv("GROQ_API_KEY", "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"),
    temperature=0.5
)
print("Chat model is ready.")

Chat model is ready.


## 3. Weather API tool + model call

**External API:** Open-Meteo — no API key.

The tool finds the city's coordinates and requests its current temperature. Then the model can decide to call the tool when asked about the weather.

In [69]:
from langchain.tools import tool
import requests

@tool
def get_weather(city: str) -> str:
    """Get the current temperature in a city using Open-Meteo."""

    place = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=20
    ).json()["results"][0]

    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m"
        },
        timeout=20
    ).json()

    return f"{city}: {data['current']['temperature_2m']} °C"


weather_model = model.bind_tools([get_weather])

question = "What is the current temperature and forecast in Hyderabad?"

response = weather_model.invoke(question)

print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': 'fc_e3823860-e2e5-453e-aba3-a6588dc67a0b', 'type': 'tool_call'}]


### Execute the requested tool and ask the model for the final answer

In [70]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage(content=question), response]

for call in response.tool_calls:
    result = get_weather.invoke(call["args"])
    print("API result:", result)
    messages.append(
        ToolMessage(content=str(result), tool_call_id=call["id"])
    )

final_answer = model.invoke(messages)
print("Final answer:", final_answer.content)

API result: Hyderabad: 28.9 °C
Final answer: The current temperature in Hyderabad is **28.9 °C**. I don’t have a forecast available at the moment, but you can check a local weather service or a weather app for the latest short‑term and long‑term predictions.



## 4. Public holidays API tool + model call

**External API:** Nager.Date — no API key.

Ask for public holidays using a country code and year.

In [52]:

import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def public_holidays(country_code: str, year: int) -> str:
    """Get public holidays using country code and year."""

    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"

    api_response = requests.get(
        url,
        headers={"Cache-Control": "no-cache"},
        timeout=20
    )

    if api_response.status_code == 204:
        return f"No holiday data returned for {country_code} in {year}."

    api_response.raise_for_status()

    data = api_response.json()

    holidays = []

    for holiday in data:
        holidays.append(
            f"{holiday['date']} - {holiday['name']}"
        )

    return "\n".join(holidays)


# Bind tool to model
tools = [public_holidays]
model_with_tools = model.bind_tools(tools)

# Ask question
question = "What are the public holidays in USA for 2026?"

llm_response = model_with_tools.invoke(
    [HumanMessage(content=question)]
)

print(llm_response.tool_calls)

# Execute tool
messages = [
    HumanMessage(content=question),
    llm_response
]

for call in llm_response.tool_calls:

    result = public_holidays.invoke(call["args"])

   # print("API Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

# Get final answer
# final_response = model.invoke(messages)

# print("Final Answer:", final_response.content)


[{'name': 'public_holidays', 'args': {'country_code': 'US', 'year': 2026}, 'id': 'fc_c3b02afc-f99b-4682-85b5-68e4874856d6', 'type': 'tool_call'}]


In [53]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(content=question),
    llm_response
]

for call in llm_response.tool_calls:

    result = public_holidays.invoke(call["args"])

    print("API result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

final_response = model.invoke(messages)

print("Final answer:", final_response.content)

API result: 2026-01-01 - New Year's Day
2026-01-19 - Martin Luther King, Jr. Day
2026-02-12 - Lincoln's Birthday
2026-02-16 - Presidents Day
2026-04-03 - Good Friday
2026-04-03 - Good Friday
2026-05-08 - Truman Day
2026-05-25 - Memorial Day
2026-06-19 - Juneteenth National Independence Day
2026-07-03 - Independence Day
2026-09-07 - Labour Day
2026-10-12 - Columbus Day
2026-10-12 - Columbus Day
2026-10-12 - Indigenous Peoples' Day
2026-11-11 - Veterans Day
2026-11-26 - Thanksgiving Day
2026-12-25 - Christmas Day
Final answer: Below is a concise list of the **federal (public) holidays** observed in the United States for 2026, along with the day of the week on which each falls. These are the days on which federal government offices, most banks, and many private‑sector employers are closed.

| Date (2026) | Day | Federal Holiday |
|-------------|-----|-----------------|
| **January 1** | Thursday | New Year’s Day |
| **January 19** | Monday | Martin Luther King Jr. Day (third Monday in Jan

## 5. Currency API tool + model call

**\*\*External API:\*\*** Frankfurter — no API key.

Ask the model to convert an amount. It can request the currency tool with the required arguments.

In [54]:
import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


@tool
def convert_currency(amount: float, source: str, target: str) -> str:
    """Convert an amount from one currency to another using exchange rates.

    Args:
        amount: The amount of money to convert, e.g. 100.
        source: 3-letter source currency code, e.g. USD.
        target: 3-letter target currency code, e.g. INR.
    """

    url = f"https://api.frankfurter.dev/v2/rate/{source}/{target}"

    api_response = requests.get(url, timeout=20)
    api_response.raise_for_status()

    data = api_response.json()
    converted_amount = amount * data["rate"]

    return (
        f"{amount} {source} = "
        f"{converted_amount:.2f} {target}. "
        f"Exchange rate: 1 {source} = {data['rate']} {target}"
    )


# Force the model to call exactly this tool, so it can't invent a wrong name
model_with_tools = model.bind_tools(
    [convert_currency],
    tool_choice="convert_currency"
)

question = "Convert 100 USD into INR."

llm_response = model_with_tools.invoke(
    [HumanMessage(content=question)]
)

print(llm_response.tool_calls)


[{'name': 'convert_currency', 'args': {'amount': 100, 'source': 'USD', 'target': 'INR'}, 'id': 'fc_765420fc-becb-4a37-83f0-d44709f060ef', 'type': 'tool_call'}]


In [55]:
messages = [
    HumanMessage(content=question),
    llm_response
]

for call in llm_response.tool_calls:

    result = convert_currency.invoke(call["args"])

    print("API Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

final_response = model.invoke(messages)

print("Final Answer:", final_response.content)

API Result: 100.0 USD = 9372.00 INR. Exchange rate: 1 USD = 93.72 INR
Final Answer: 100 USD = **9,372 INR** (exchange rate: 1 USD ≈ 93.72 INR).


## 6. Country API tool + model call

**External API:** REST Countries — no API key.

Ask for the capital and population of a country.

In [56]:
import requests
from langchain_core.tools import tool

@tool
def country_details(country: str) -> str:
    """Get country name, capital, region and population."""

    url = f"https://api.restcountries.com/countries/v5/names.common/{country}"

    response = requests.get(
        url,
        headers={
            "Authorization": "Bearer rc_live_demo"
        },
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    countries = data["data"]["objects"]

    if not countries:
        return f"Country not found: {country}"

    country_data = countries[0]

    return (
        f"Country: {country_data['names']['common']}; "
        f"Capital: {country_data['capitals'][0]}; "
        f"Region: {country_data['region']}; "
        f"Population: {country_data['population']:,}"
    )

result = country_details.invoke({
    "country": "India"
})

print(result)


tools = [country_details]

model_with_tools = model.bind_tools(tools)

question = "Tell me about India."

llm_response = model_with_tools.invoke(
    [HumanMessage(content=question)]
)

print(llm_response.tool_calls)

Country: Canada; Capital: {'attributes': {'administrative': False, 'constitutional': False, 'executive': False, 'judicial': False, 'legislative': False, 'primary': True}, 'coordinates': {'lat': 45.42, 'lng': -75.7}, 'name': 'Ottawa'}; Region: Americas; Population: 41,798,407
[{'name': 'country_details', 'args': {'country': 'India'}, 'id': 'fc_451518f8-7453-4078-a754-f68e42a7f70b', 'type': 'tool_call'}]


In [21]:
messages = [
    HumanMessage(content=question),
    llm_response
]

for call in llm_response.tool_calls:

    result = country_details.invoke(call["args"])

    print("API Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

final_response = model.invoke(messages)

print("Final Answer:", final_response.content)

API Result: Country: Canada; Capital: {'attributes': {'administrative': False, 'constitutional': False, 'executive': False, 'judicial': False, 'legislative': False, 'primary': True}, 'coordinates': {'lat': 45.42, 'lng': -75.7}, 'name': 'Ottawa'}; Region: Americas; Population: 41,798,407
Final Answer: The tool returned information for Canada instead of India. I will provide accurate information about India based on general knowledge.

**India** is a country in South Asia. Here are some key facts:

*   **Capital:** New Delhi
*   **Region:** South Asia
*   **Population:** Approximately 1.4 billion people, making it the most populous country in the world (as of 2023, surpassing China).
*   **Official Languages:** Hindi and English (along with 22 other scheduled languages).
*   **Currency:** Indian Rupee (INR)
*   **Government:** Federal parliamentary democratic republic.
*   **Geography:** Bordered by the Indian Ocean to the south, the Arabian Sea to the southwest, and the Bay of Bengal to

## 7. Random joke API tool + model call

**External API:** Official Joke API — no API key.

Ask the model to get a random joke.

In [57]:
import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def get_random_joke() -> str:
    """Get a random joke from JokeAPI."""

    api_response = requests.get(
        "https://v2.jokeapi.dev/joke/Any?type=twopart",
        timeout=20
    )

    api_response.raise_for_status()
    data = api_response.json()

    return f"{data['setup']} — {data['delivery']}"
tools = [get_random_joke]

joke_model = model.bind_tools(tools)

question = "Tell me a random joke. Use the get_random_joke tool. but joke neddded to be lugh"

response = joke_model.invoke(
    [HumanMessage(content=question)]
)

print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'get_random_joke', 'args': {'': {}}, 'id': 'fc_34cd84fb-3164-4b83-9f19-61ae1c358c6a', 'type': 'tool_call'}]


In [58]:
messages = [
    HumanMessage(content=question),
    response
]

for call in response.tool_calls:

    result = get_random_joke.invoke(call["args"])

    print("API result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

print("Final answer:", model.invoke(messages).content)

API result: What's the best thing about Switzerland? — I don't know, but the flag is a big plus.
Final answer: Here’s a random joke for you:

**What’s the best thing about Switzerland? — I don’t know, but the flag is a big plus.**


## 8. Random user API tool + model call

**External API:** Random User Generator — no API key.

This API generates synthetic sample user information, not real people's details.

In [59]:
import requests
from langchain_core.tools import tool

@tool
def get_random_user() -> str:
    """Generate a sample user's name, email, and country."""

    api_response = requests.get(
        "https://randomuser.me/api/",
        timeout=20
    )

    api_response.raise_for_status()
    data = api_response.json()["results"][0]

    person = data["name"]

    return (
        f"Name: {person['first']} {person['last']}; "
        f"Email: {data['email']}; "
        f"Country: {data['location']['country']}"
    )
tools = [get_random_user]

user_model = model.bind_tools(tools)

question = "Use the get_random_user tool to generate a sample user profile."

response = user_model.invoke(question)

print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'get_random_user', 'args': {'': {}}, 'id': 'fc_cf0ab4a4-fb2d-4ee9-afe6-8a8a5b7fe19c', 'type': 'tool_call'}]


In [60]:
messages = [HumanMessage(content=question), response]

for call in response.tool_calls:
    result = get_random_user.invoke(call["args"])
    print("API result:", result)
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

print("Final answer:", model.invoke(messages).content)

API result: Name: Alexis Deschamps; Email: alexis.deschamps@example.com; Country: Switzerland
Final answer: Here’s a sample user profile generated for you:

- **Name:** Alexis Deschamps  
- **Email:** alexis.deschamps@example.com  
- **Country:** Switzerland


## 9. Dog API tool + model call

**External API:** Dog CEO — no API key.

Ask for a random dog image URL.

In [61]:
@tool
def get_dog_image() -> str:
    "Get a URL for a random dog image."
    data = requests.get(
        "https://dog.ceo/api/breeds/image/random",
        timeout=20
    ).json()

    return data["message"]

dog_model = model.bind_tools([get_dog_image])

question = "Give me a random dog image URL using the dog tool."
response = dog_model.invoke(question)
print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'get_dog_image', 'args': {'': {}}, 'id': 'fc_3909cdef-1d69-4b06-bbb0-ca437f471d3a', 'type': 'tool_call'}]


In [62]:
messages = [HumanMessage(content=question), response]

for call in response.tool_calls:
    result = get_dog_image.invoke(call["args"])
    print("API result:", result)
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

print("Final answer:", model.invoke(messages).content)

API result: https://images.dog.ceo/breeds/mix/mydogggg_2025-11-06.jpg
Final answer: Here’s a random dog image for you:

https://images.dog.ceo/breeds/mix/mydogggg_2025-11-06.jpg


## 10. Combine all seven tools and use one model

Now bind all seven tools to the same model. The model can choose which tool is appropriate for a question.

In [63]:
tools = [
    get_weather,
    convert_currency,
    country_details,
    public_holidays,
    get_random_joke,
    get_random_user,
    get_dog_image
]

all_tools_model = model.bind_tools(tools)

question = "What is the capital of Japan and what is the current weather in Hyderabad?"
response = all_tools_model.invoke(question)

print("Selected tool calls:", response.tool_calls)

Selected tool calls: [{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': 'fc_12bbdd93-b354-47cd-9c24-e8687f7c2cc5', 'type': 'tool_call'}]


## 11. Execute the model's tool calls

This small loop executes whichever tools the model requested and sends their results back to the model.

In [64]:
tool_map = {item.name: item for item in tools}

messages = [HumanMessage(content=question), response]

for call in response.tool_calls:
    selected_tool = tool_map[call["name"]]
    result = selected_tool.invoke(call["args"])
    print(call["name"], "=>", result)

    messages.append(
        ToolMessage(content=str(result), tool_call_id=call["id"])
    )

final_answer = model.invoke(messages)
print("\\nFinal answer:")
print(final_answer.content)

get_weather => Hyderabad: 28.9 °C
\nFinal answer:
The capital of Japan is **Tokyo**.

The current weather in Hyderabad is **28.9 °C**.


## flights api

In [65]:
import requests
from langchain_core.tools import tool

@tool
def search_flights(country: str) -> str:
    """Get currently tracked aircraft information for a country."""

    response = requests.get(
        "https://opensky-network.org/api/states/all",
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    aircraft = data.get("states") or []

    results = []

    for flight in aircraft:
        if flight[2] == country:
            results.append(
                f"Callsign: {flight[1]}\n"
                f"Country: {flight[2]}\n"
                f"Longitude: {flight[5]}\n"
                f"Latitude: {flight[6]}"
            )

        if len(results) == 5:
            break

    return "\n\n".join(results) if results else "No aircraft found."


flight_model = model.bind_tools([search_flights])

question = "Find currently tracked aircraft originating from India. Use the flight tool."

response = flight_model.invoke(question)

print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'search_flights', 'args': {'country': 'India'}, 'id': 'fc_2d6a2941-aa89-40cd-9ac5-d99642ddbc35', 'type': 'tool_call'}]


In [66]:
messages = [
    HumanMessage(content=question),
    response
]

for call in response.tool_calls:

    result = search_flights.invoke(call["args"])

    print("API result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

final_response = model.invoke(messages)

print("Final answer:", final_response.content)

API result: Callsign: AXB209  
Country: India
Longitude: 56.8825
Latitude: 24.6433

Callsign: AIC8GW  
Country: India
Longitude: 74.9014
Latitude: 26.0326

Callsign: AIC6FW  
Country: India
Longitude: 87.1935
Latitude: 26.2115

Callsign: AXB1059 
Country: India
Longitude: 77.6956
Latitude: 28.0746

Callsign: AIC2GM  
Country: India
Longitude: 77.7602
Latitude: 13.8699
Final answer: **Currently tracked aircraft that originate from India** (as returned by the flight‑tracking tool):

| Callsign | Country | Latitude (°) | Longitude (°) |
|----------|---------|--------------|---------------|
| AXB209   | India   | 24.6433      | 56.8825       |
| AIC8GW   | India   | 26.0326      | 74.9014       |
| AIC6FW   | India   | 26.2115      | 87.1935       |
| AXB1059  | India   | 28.0746      | 77.6956       |
| AIC2GM   | India   | 13.8699      | 77.7602       |

These are the aircraft that the flight‑tracking system currently has on its radar and that are listed with **India** as their country o

## Important concepts

| Code | Purpose |
|---|---|
| `ChatGroq()` | Creates the chat model |
| `@tool` | Converts a Python function into a tool |
| `bind_tools()` | Makes tool definitions available to the model |
| `invoke()` | Sends a question to the model or executes a tool |
| `tool_calls` | Shows the tool requests made by the model |
| `ToolMessage` | Sends tool output back into the conversation |

**Important distinction:** `bind_tools()` does not execute the external API. The model requests a tool call; your Python code executes it and returns the result. For a production workflow, use LangChain's `create_agent()` to manage the tool-calling loop.

## Official documentation and API references

- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain Models](https://docs.langchain.com/oss/python/langchain/models)
- [Open-Meteo](https://open-meteo.com/)
- [Frankfurter](https://frankfurter.dev/)
- [REST Countries](https://restcountries.com/)
- [Nager.Date](https://date.nager.at/)
- [Official Joke API](https://official-joke-api.appspot.com/)
- [Random User Generator](https://randomuser.me/)
- [Dog CEO API](https://dog.ceo/dog-api/)

All seven external APIs are designed to work without API keys. Groq requires its own API key for the chat model.